# 06 — EDA Summary & Dataset Decision

## Problem

Input: social graph `G=(V,E)`

PYMK: generate potential connection candidates

Trust filtering: remove suspicious/untrusted candidates

Output: top-K no-fraud / trust-aware connection recommendations

> Bot labels are used as a suspicious-account proxy, not as fraud ground truth.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy import sparse
from scipy.sparse.csgraph import connected_components

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "MGTAB"
RESULTS_DIR = PROJECT_ROOT / "results" / "eda_05"
if not (RESULTS_DIR / "candidate_report.csv").is_file():
    raise FileNotFoundError("Hãy chạy 05_fraud_aware_eda.ipynb trước để tạo results/eda_05/.")

In [2]:
def load_tensor(name):
    try:
        return torch.load(DATA_DIR / name, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(DATA_DIR / name, map_location="cpu")

edge_index = load_tensor("edge_index.pt").long()
edge_type = load_tensor("edge_type.pt").long()
features = load_tensor("features.pt").float()
labels_bot = load_tensor("labels_bot.pt").long().numpy()
src = edge_index[0].numpy()
dst = edge_index[1].numpy()
types = edge_type.numpy()
num_nodes = features.shape[0]
num_edges = edge_index.shape[1]
candidate_report = pd.read_csv(RESULTS_DIR / "candidate_report.csv")
aggregate_report = pd.read_csv(RESULTS_DIR / "aggregate_report.csv")
edge_mixing = pd.read_csv(RESULTS_DIR / "edge_mixing.csv")

def aggregate_value(metric):
    return float(aggregate_report.loc[aggregate_report["metric"] == metric, "value"].iloc[0])


## Why MGTAB

MGTAB phù hợp cho PoC này vì là dataset công khai, có account nodes, graph nhiều quan hệ, nhãn human/bot và cùng node space cho Recommendation + Trust components. Đây là lựa chọn thực nghiệm cho PoC, không phải bằng chứng rằng dữ liệu mô phỏng đầy đủ một hệ thống fraud thực tế.

In [3]:
overview = pd.DataFrame({
'Property': ['Nodes', 'Edge rows', 'Features', 'Relations', 'Friend edge rows', 'Human', 'Bot'],
'Value': [
        num_nodes, num_edges, features.shape[1], types.max() + 1,
        int((types == 1).sum()), int((labels_bot == 0).sum()), int((labels_bot == 1).sum()),
    ],
})
overview

,Property,Value
0,Nodes,10199
1,Edge rows,1700108
2,Features,788
3,Relations,7
4,Friend edge rows,412575
5,Human,7451
6,Bot,2748


In [4]:
pair_keys = np.minimum(src, dst).astype(np.int64) * num_nodes + np.maximum(src, dst).astype(np.int64)
quality_summary = pd.DataFrame({
    "Check": ["Missing raw files", "Invalid node IDs", "Duplicate undirected pairs", "Self-loops", "Feature NaN/Inf", "Unexpected relation IDs"],
    "Result": [
        int(sum(not (DATA_DIR / name).is_file() for name in ["edge_index.pt", "edge_type.pt", "edge_weight.pt", "features.pt", "labels_bot.pt", "labels_stance.pt"])),
        bool((src.min() < 0) or (dst.min() < 0) or (src.max() >= num_nodes) or (dst.max() >= num_nodes)),
        int(len(pair_keys) - np.unique(pair_keys).size),
        int((src == dst).sum()),
        bool(torch.isnan(features).any() or torch.isinf(features).any()),
        sorted(set(types.tolist()) - set(range(7))),
    ],
})
quality_summary["Planned Action"] = ["None", "Investigate before modeling", "Document; deduplicate in baseline if appropriate", "Investigate semantics", "Must clean before modeling", "Investigate mapping"]
quality_summary

,Check,Result,Planned Action
0,Missing raw files,0,None
1,Invalid node IDs,False,Investigate before modeling
2,Duplicate undirected pairs,791648,Document; deduplicate in baseline if appropriate
3,Self-loops,205819,Investigate semantics
4,Feature NaN/Inf,False,Must clean before modeling
5,Unexpected relation IDs,[],Investigate mapping


In [5]:
row = np.concatenate([src, dst])
col = np.concatenate([dst, src])
adj = sparse.csr_matrix((np.ones(len(row), dtype=np.int8), (row, col)), shape=(num_nodes, num_nodes))
adj.data[:] = 1
adj.eliminate_zeros()
degree = np.diff(adj.indptr)
n_components, component_ids = connected_components(adj, directed=False)
component_sizes = np.bincount(component_ids)
directed_keys = src.astype(np.int64) * num_nodes + dst.astype(np.int64)
reverse_keys = dst.astype(np.int64) * num_nodes + src.astype(np.int64)
graph_summary = pd.DataFrame({
    "Metric": ["Mean degree", "Median degree", "P90 degree", "P95 degree", "Max degree", "Density", "Components", "Largest component share", "Reciprocity"],
    "Value": [
        degree.mean(), np.median(degree), np.percentile(degree, 90), np.percentile(degree, 95), degree.max(),
        np.unique(pair_keys).size / (num_nodes * (num_nodes - 1) / 2), n_components, component_sizes.max() / num_nodes,
        np.isin(reverse_keys, directed_keys).mean(),
    ],
})
graph_summary

,Metric,Value
0,Mean degree,177.597412
1,Median degree,114.000000
2,P90 degree,401.000000
3,P95 degree,562.000000
4,Max degree,5009.000000
5,Density,0.017469
6,Components,55.000000
7,Largest component share,0.994705
8,Reciprocity,0.554716


In [6]:
pymk_summary = pd.DataFrame({
    "Metric": ["Mean candidate count", "Median candidate count", "P90 candidate count", "P95 candidate count", "Users with candidate", "Users without candidate"],
    "Value": [
        aggregate_value("candidate_count_mean"), aggregate_value("candidate_count_median"),
        aggregate_value("candidate_count_p90"), aggregate_value("candidate_count_p95"),
        aggregate_value("human users with >=1 candidate") / len(candidate_report),
        aggregate_value("human users with no candidate") / len(candidate_report),
    ],
})
pymk_summary

,Metric,Value
0,Mean candidate count,7424.250302
1,Median candidate count,7901.000000
2,P90 candidate count,8910.000000
3,P95 candidate count,9065.000000
4,Users with candidate,0.999463
5,Users without candidate,0.000537


In [7]:
fraud_summary = pd.DataFrame({
    "Metric": ["Bot prevalence", "Human users with >=1 bot candidate", "Mean bot candidate count", "Median bot candidate count", "Mean bot ratio", "Median bot ratio"],
    "Value": [
        float((labels_bot == 1).mean()),
        aggregate_value("percentage with >=1 bot candidate"),
        aggregate_value("bot_candidate_count_mean"),
        aggregate_value("bot_candidate_count_median"),
        aggregate_value("bot_candidate_ratio_mean"),
        aggregate_value("bot_candidate_ratio_median"),
    ],
})
fraud_summary

,Metric,Value
0,Bot prevalence,0.269438
1,Human users with >=1 bot candidate,0.999463
2,Mean bot candidate count,1848.531606
3,Median bot candidate count,1962.000000
4,Mean bot ratio,0.246160
5,Median bot ratio,0.249458


## End-to-end interpretation

```text
MGTAB
  ↓
Friend / undirected projection
  ↓
2-hop candidate generation
  ↓
Candidate set containing human + bot accounts
  ↓
Trust/Fraud Agent (bot as suspicious proxy)
  ↓
Filtered candidate set
  ↓
Future ranking / Top-K
```

EDA mới validate data và problem setup; chưa validate model performance.

## Dataset limitations

1. MGTAB là Twitter/social-media dataset.
2. Friend/follow relation không hoàn toàn tương đương Facebook-style mutual friendship.
3. Bot != fraud.
4. Dataset không được tạo riêng cho PYMK.
5. Symmetrize graph có thể làm mất thông tin direction.
6. Candidate generation bị ảnh hưởng bởi hub và không tạo candidate giữa các component tách biệt.

In [8]:
quality_ok = not bool(quality_summary.loc[quality_summary["Check"].isin(["Missing raw files", "Invalid node IDs", "Feature NaN/Inf", "Unexpected relation IDs"]), "Result"].astype(bool).any())
candidate_coverage = aggregate_value("human users with >=1 candidate") / len(candidate_report)
if quality_ok and candidate_coverage > 0:
    print("## Dataset Decision")
    print("MGTAB is selected as the primary dataset for the current PoC.")
    print("Reasons: graph structure supports 2-hop candidate generation; labels provide a suspicious-account proxy; and the dataset has a shared node space for recommendation and trust analysis.")
    print("This decision is conditional on the limitations above and does not claim fraud ground truth.")
else:
    print("## Dataset Decision")
    print("MGTAB is not yet ready to be selected as the primary dataset; resolve the failed quality checks or candidate coverage first.")

## Dataset Decision
MGTAB is selected as the primary dataset for the current PoC.
Reasons: graph structure supports 2-hop candidate generation; labels provide a suspicious-account proxy; and the dataset has a shared node space for recommendation and trust analysis.
This decision is conditional on the limitations above and does not claim fraud ground truth.


## Next step

Sau khi chốt dataset:

```text
MGTAB raw → Data Preparation → Canonical Schema
→ Link Prediction Experimental Setup
→ Common Neighbors / Jaccard / Adamic-Adar / Preferential Attachment
→ Evaluation → Trust Filtering Integration
```

Notebook 06 không implement baseline, GNN, ML, API hay model training.